# 推理瓶颈探索性分析

本notebook对单细胞基础模型进行推理profiling，分析各步骤的时间分解和内存占用。

In [ ]:
import sys
sys.path.insert(0, "..")
import torch
import numpy as np
from scinfer.evaluation import InferenceProfiler
from scinfer.utils.gpu import get_gpu_info

In [ ]:
# 使用合成Transformer模型演示profiling流程
# 实际使用时替换为真实模型：
# from scinfer.adapters import ScGPTAdapter
# adapter = ScGPTAdapter.from_pretrained()
from scripts.profile_models import create_synthetic_model
model = create_synthetic_model(model_type="transformer", n_layers=6, hidden_size=512)

In [ ]:
profiler = InferenceProfiler(model=model, device="cuda" if torch.cuda.is_available() else "cpu")
results = profiler.profile(input_shape=(4, 1200, 512))

In [ ]:
import matplotlib.pyplot as plt
categories = list(results.get("time_breakdown", {}).keys())
times = list(results.get("time_breakdown", {}).values())
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(categories, times, color=["#2196F3", "#FF9800", "#4CAF50", "#9C27B0", "#F44336"])
ax.set_ylabel("Time (ms)")
ax.set_title("Inference Time Breakdown by Component")
plt.tight_layout()
plt.show()

In [ ]:
memory = results.get("memory_breakdown", {})
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(list(memory.keys()), list(memory.values()), color="#2196F3")
ax.set_xlabel("Memory (MB)")
ax.set_title("Memory Usage Breakdown by Layer")
plt.tight_layout()
plt.show()

In [ ]:
input_lengths = [128, 512, 2048, 4096]
# Placeholder data for demonstration
time_by_length = {l: {"attention": np.random.uniform(5, 50), "ffn": np.random.uniform(10, 80), "other": np.random.uniform(2, 10)} for l in input_lengths}
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(input_lengths))
width = 0.25
ax.bar(x - width, [time_by_length[l]["attention"] for l in input_lengths], width, label="Attention")
ax.bar(x, [time_by_length[l]["ffn"] for l in input_lengths], width, label="FFN")
ax.bar(x + width, [time_by_length[l]["other"] for l in input_lengths], width, label="Other")
ax.set_xticks(x)
ax.set_xticklabels(input_lengths)
ax.set_xlabel("Input Length (genes)")
ax.set_ylabel("Time (ms)")
ax.set_title("Bottleneck Changes Across Input Lengths")
ax.legend()
plt.tight_layout()
plt.show()

## 关键发现

- FFN计算通常是主要瓶颈...
- 注意力计算随序列长度平方增长...
- 小模型（10M-316M）的瓶颈分布与LLM（7B+）显著不同...